# Exercise Sheet 2: Causality

---

**Data for Exercise 3-1** (Chicken vs. Egg production)

| Year | Chicken (Y) | Egg (X) |
|------|-------------|---------|
| 1930 | 468491     | 3581   |
| 1931 | 449743     | 3532   |
| 1932 | 436815     | 3327   |
| 1933 | 444523     | 3255   |
| 1934 | 433937     | 3156   |
| 1935 | 389958     | 3081   |


## Exercise 3-1: Granger causal test

### a) Auto-regression for X and Y (lag = 1)

The model is:

$$
\begin{pmatrix}
Y^{(2)} \\
\vdots \\
Y^{(n)}
\end{pmatrix}
= \beta_0 \begin{pmatrix} 1 \\ \vdots \\ 1 \end{pmatrix}
+ \beta_1 \begin{pmatrix} Y^{(1)} \\ \vdots \\ Y^{(n-1)} \end{pmatrix}
+ \beta_2 \begin{pmatrix} X^{(1)} \\ \vdots \\ X^{(n-1)} \end{pmatrix}
+ \epsilon
$$

**Data for regression (5 observations)**:

- $Y^{(n)}$ = [449743, 436815, 444523, 433937, 389958]  
- $Y^{(n-1)}$ = [468491, 449743, 436815, 444523, 433937]  
- $X^{(n-1)}$ = [3581, 3532, 3327, 3255, 3156]


In [1]:
import numpy as np

# Data
y_n = np.array([449743, 436815, 444523, 433937, 389958])
y_lag = np.array([468491, 449743, 436815, 444523, 433937])
x_lag = np.array([3581, 3532, 3327, 3255, 3156])

# Full design matrix
X_full = np.column_stack((np.ones(5), y_lag, x_lag))

# OLS solution
beta_full = np.linalg.lstsq(X_full, y_n, rcond=None)[0]
y_hat_full = X_full @ beta_full
rss1 = np.sum((y_n - y_hat_full)**2)

print("β̂ =", beta_full)
print("RSS₁ (full model) =", rss1)
print("Fitted values:", np.round(y_hat_full, 4))


β̂ = [ 1.20111885e+05 -6.90430740e-02  1.01396054e+02]
RSS₁ (full model) = 1023104023.6108685
Fitted values: [450865.0948 447191.1077 427297.5055 419464.8056 410157.4863]


### b) Auto-regression without Egg feature (only AR(1) on Chicken)

Design matrix without egg feature:

$$
X_0 = \begin{pmatrix}
1 & Y^{(n-1)}_1 \\
\vdots & \vdots \\
1 & Y^{(n-1)}_5
\end{pmatrix}
$$


In [2]:
# Reduced model (no X_lag)
X_reduced = np.column_stack((np.ones(5), y_lag))
beta_reduced = np.linalg.lstsq(X_reduced, y_n, rcond=None)[0]
y_hat_reduced = X_reduced @ beta_reduced
rss2 = np.sum((y_n - y_hat_reduced)**2)

print("β̂₀ =", beta_reduced)
print("RSS₂ (reduced model) =", rss2)


β̂₀ = [-5.17190470e+04  1.08061854e+00]
RSS₂ (reduced model) = 1385892406.5564842


### c) Apply statistical test (Granger statistic)

$$
GS = \frac{(RSS_2 - RSS_1)/d}{RSS_2/(n-2d)}
$$

with $d=1$, $n=5$.

Critical value: $F_{0.05}(1,3) = 10.13$


In [3]:
# Granger statistic
n = 5
d = 1
gs = ((rss2 - rss1) / d) / (rss2 / (n - 2 * d))
print("Granger statistic GS =", round(gs, 4))

# Interpretation
print("GS < 10.13 → Eggs do NOT Granger-cause Chickens")


Granger statistic GS = 0.7853
GS < 10.13 → Eggs do NOT Granger-cause Chickens


### d) Causal test in the opposite direction (Chicken → Egg)

Model:
$$
X^{(n)} = \beta_0 + \beta_1 X^{(n-1)} + \beta_2 Y^{(n-1)} + \epsilon
$$


In [4]:
# Data for reverse direction
x_n = np.array([3532, 3327, 3255, 3156, 3081])
x_lag_rev = np.array([3581, 3532, 3327, 3255, 3156])  # same as before
y_lag_rev = y_lag.copy()  # previous chicken

# Full model for eggs
X_full_rev = np.column_stack((np.ones(5), x_lag_rev, y_lag_rev))
beta_full_rev = np.linalg.lstsq(X_full_rev, x_n, rcond=None)[0]
x_hat_full = X_full_rev @ beta_full_rev
rss1_rev = np.sum((x_n - x_hat_full)**2)

# Reduced model (only AR(1) on eggs)
X_reduced_rev = np.column_stack((np.ones(5), x_lag_rev))
beta_reduced_rev = np.linalg.lstsq(X_reduced_rev, x_n, rcond=None)[0]
x_hat_reduced = X_reduced_rev @ beta_reduced_rev
rss2_rev = np.sum((x_n - x_hat_reduced)**2)

gs_rev = ((rss2_rev - rss1_rev) / d) / (rss2_rev / (n - 2 * d))

print("β̂ (Chicken → Egg) =", beta_full_rev)
print("RSS₁_rev =", rss1_rev)
print("RSS₂_rev =", rss2_rev)
print("GS (reverse) =", round(gs_rev, 4))
print("GS < 10.13 → Chickens do NOT Granger-cause Eggs")


β̂ (Chicken → Egg) = [-9.41423258e+02  5.71643364e-01  5.11542777e-03]
RSS₁_rev = 8511.149774127953
RSS₂_rev = 13768.290215049534
GS (reverse) = 1.1455
GS < 10.13 → Chickens do NOT Granger-cause Eggs


**Conclusion** (as in the sheet):  
Since neither direction shows Granger causality (with only 5 usable rows), we cannot determine which came first — chicken or egg. With more data the test would likely become decisive.


## Exercise 3-2: Multivariate Granger model

### a) Is the graphical Granger model with adaptive lasso penalty λₙ = n³/² consistent?

**Answer**: No, it is **inconsistent**.

Adaptive lasso is consistent only if both conditions hold:

$$
\frac{\lambda_n}{\sqrt{n}} \to 0 \quad \text{and} \quad \lambda_n n^{\frac{\omega-1}{2}} \to \infty
$$

Plug in $\lambda_n = n^{3/2}$:

- First term: $n^{3/2} / \sqrt{n} = n \not\to 0$ → **violated**
- Second term: $n^{3/2} \cdot n^{\frac{\omega-1}{2}} = n^{\frac{\omega+2}{2}} \to \infty$ (ok)

Because the first condition fails, the estimator is inconsistent.


### b) What is the HMMLGA algorithm for, and what are its hyperparameters?

**HMMLGA** (Heterogeneous Minimum Message Length Genetic Algorithm) is a genetic-algorithm-based search that explores possible parent sets $Q_i$ for each time series. It uses the **Minimum Message Length (MML)** criterion (combined with maximum-likelihood) to score candidate causal graphs and improves them via crossover and mutation.

**Reference**: Hlaváčková-Schindler K, Plant C. *Heterogeneous Graphical Granger Causality by Minimum Message Length*. Entropy. 2020.

**Hyperparameters**:
- Population size ($m$)
- Maximum number of generations ($n_g$)
- Mutation probability
- Lag parameter ($d$)
- Ridge regularization parameter ($\lambda$)


## Exercise 3-3: Bivariate causal models on non-temporal data

### a) Why is the causal relationship non-identifiable with bivariate additive noise models in the linear Gaussian case?

Linear-Gaussian additive noise models are **symmetric**:

$$
Y = aX + N, \quad N \perp X
$$
$$
X = bY + N', \quad N' \perp Y
$$

Both directions fit the data equally well. Without non-linearity, interventions, or additional assumptions, the direction cannot be identified.


### b) Recall Example 1 from lecture 4

1. **Causal relation**: age (X) → diastolic blood pressure (Y)  
   → **X → Y** (makes perfect sense biologically).

2. **How to change P(effect|cause) without affecting P(cause)**:  
   Intervene on blood pressure (e.g., prescribe medication, encourage sport/diet). This changes $P(Y|X)$ for every age while $P(X)$ (age distribution) remains unchanged.

3. **Anti-causal direction (Y → X)**:  
   Impossible in reality. Changing blood pressure cannot alter a person's age, and we cannot keep the age distribution inside each blood-pressure group fixed while manipulating blood pressure. The anti-causal direction makes no physical sense.


In [6]:
# summary table of results
import pandas as pd
results = pd.DataFrame({
    "Direction": ["Egg → Chicken", "Chicken → Egg"],
    "RSS_full": [rss1, rss1_rev],
    "RSS_reduced": [rss2, rss2_rev],
    "Granger Statistic": [gs, gs_rev],
    "Conclusion": ["No causality", "No causality"]
})
print(results)


       Direction      RSS_full   RSS_reduced  Granger Statistic    Conclusion
0  Egg → Chicken  1.023104e+09  1.385892e+09           0.785317  No causality
1  Chicken → Egg  8.511150e+03  1.376829e+04           1.145489  No causality
